# 📗 02. 데이터 EDA — H&M 커머스 탐색적 분석

> 엔코아 AI캠퍼스 · 데이터 분석 & AI 머신러닝 캠프

이 노트북은 **01_데이터이해_전처리**가 만든 정제본으로 **탐색적 데이터 분석(EDA)** 을 합니다. day07(EDA·시각화) 전체와 day08(기술통계)을 이 파트에서 다룹니다 — **추론통계(가설검정·회귀)는 03_데이터_통계**에서 이어집니다.

**이 노트북 읽는 법**
- 각 절 시작에 **"이 절에서 할 일"** 한 줄이 있습니다.
- 이 노트북에는 **완성된 코드가 없습니다.** 각 절이 **무엇을 만들어야 하는지**와 **어떤 도구(함수·핵심 인자)를 쓰는지**를 알려 주니, **코드는 그것을 보고 스스로 쓰세요.**
- 막히면 **"막히면 볼 곳"** 이 가리키는 가이드 절로 가서 **어떤 도구를 쓰는지** 확인하세요.
- 각 절 끝에 **"여기까지 되면 통과"** 로 스스로 확인하세요(자가채점 없음).
- **3절(필수 미션)은 반드시**, **4절(선택 카탈로그)은 원하는 것만** — 이 둘은 성격이 다르니 헷갈리지 마세요.

**목차**
1. 정제본 불러오기
2. 가이드 — 집계·시각화·기술통계 도구
3. 필수 EDA 미션 (반드시)
4. 선택 EDA 미션 카탈로그 (골라서)
5. 관찰 정리 (서술)

## 1. 정제본 불러오기
**이 절에서 할 일**: 01 이 저장한 정제본(`output/hm_clean.csv`)을 **직접 불러와**, 이후 모든 절에서 이어 쓸 `df` 와 표본 `df_s`, 파생변수 세 개를 준비합니다.

> ⚠️ **01 을 먼저 끝내야 합니다.** 이 노트북은 01 의 마지막 셀이 저장한 정제본에서 시작합니다. `output/hm_clean.csv` 가 없으면 01 로 돌아가 저장 셀까지 실행하고 오세요. 원본 3개 테이블을 여기서 다시 읽지는 않습니다.

**이 노트북이 쓰는 컬럼** — 01 의 정제본에 아래가 남아 있어야 뒤 절의 지침을 그대로 따를 수 있습니다: `t_dat`·`customer_id`·`article_id`·`price`·`age`·`sales_channel_id`·`club_member_status`·`fashion_news_frequency`·`product_group_name`·`prod_name`·`index_group_name`. 빠진 것이 있으면 01 에서 그 컬럼을 남긴 채 다시 저장하세요.

> 🔧 01 에서 여러분이 고른 처리 규칙(연령 범위·0원 거래·채널 코드·멤버십 결측)이 그대로 반영되므로, 아래 지침의 "이렇게 나오면 맞다"와 **행수·수치가 조금 달라도 정상**입니다.

**1-1. 준비 — 라이브러리와 한글 폰트**

- **무엇을 만드나**: 이 노트북에서 쓸 라이브러리를 한 셀에 모아 불러오고, 그래프의 한글이 깨지지 않게 폰트를 설정합니다.
- **왜**: matplotlib 의 기본 폰트에는 한글 글자가 없어서, 폰트를 지정하지 않으면 그래프의 모든 한글이 네모(□)로 나옵니다.
- **어떤 도구를 쓰나**:
  - `pandas`(`pd`)·`numpy`(`np`)·`matplotlib.pyplot`(`plt`)·`seaborn`(`sns`) — **별칭은 이 관례 그대로** 쓰세요(뒤 절 지침이 이 이름으로 설명합니다).
  - `from scipy import stats` — 2-4 절의 절사평균·왜도·첨도에서 씁니다.
  - 한글 폰트: `platform.system()` 으로 OS 를 확인해 Windows 는 `'Malgun Gothic'`, macOS(`'Darwin'`)는 `'AppleGothic'`, 그 외(Colab 등)는 `'NanumGothic'` 을 `plt.rcParams['font.family']` 에 넣습니다.
  - 음수 눈금이 네모로 깨지지 않게 `plt.rcParams['axes.unicode_minus'] = False`.
  - seaborn 테마를 쓰려면 `sns.set_theme(style=…, font=…, rc={'axes.unicode_minus': False})` 처럼 폰트를 함께 넘깁니다.
  - 경고 문구가 시끄러우면 `warnings.filterwarnings('ignore')`.
- **이렇게 나오면 맞다**: 에러 없이 셀이 끝나고, 뒤에서 그린 그래프의 한글 제목이 네모(□) 없이 보입니다.
- **함정**: `sns.set_theme()` 은 `font.family` 를 자기 기본값으로 **되돌립니다** — 폰트를 먼저 지정하고 그 뒤에 `set_theme()` 을 부르면 한글이 다시 깨집니다. 순서를 바꾸거나 `set_theme(font=…)` 로 함께 넘기세요.

In [ ]:
# 여기에 코드를 작성하세요

**1-2. 정제본 불러와 `df` 만들기**

- **무엇을 만드나**: 정제본 CSV(`output/hm_clean.csv`)를 읽어 **`df`** 라는 이름으로 두고, 날짜 컬럼 `t_dat` 을 날짜형으로 되돌린 뒤 **데이터를 먼저 훑습니다** — 모양·앞부분·자료형·요약 순서로.
- **왜**: 이 노트북의 모든 절이 **`df`** 라는 이름을 쓴다고 가정합니다. 이름을 다르게 두면 뒤 절의 지침을 그대로 따를 수 없습니다.
- **어떤 도구를 쓰나**:
  - 읽기: `pd.read_csv(<경로>)` — 경로는 위의 정제본 경로 그대로.
  - 날짜형 변환: `pd.to_datetime(<컬럼>)` 의 결과를 같은 컬럼에 다시 넣습니다.
  - 훑기 4종: 모양 `df.shape` → 앞부분 `display(df.head(3))` → 열·자료형·결측 `df.info()` → 수치 요약 `df.describe()` (DataFrame 은 `print` 보다 `display` 가 표로 예쁘게 나옵니다).
  - 범주형 열은 `df.describe()` 에 안 나옵니다 — `df.describe(exclude='number')` 로 개수·고유값 수·최빈값을 봅니다.
- **이렇게 나오면 맞다**: `df.shape` 가 (10만 행 안팎, 30여 컬럼)처럼 찍히고 head·info·describe 표가 보이며, `df['t_dat'].dtype` 이 `datetime64[ns]` 입니다.
- **함정**: CSV 로 저장했다가 다시 읽으면 **날짜형이 문자열로 풀립니다.** 변환을 빠뜨리면 뒤에서 `.dt.month` 를 쓸 때 `Can only use .dt accessor with datetimelike values` 에러가 납니다.

In [ ]:
# 여기에 코드를 작성하세요

> ⚠️ **발제문 설명과 데이터가 다르면, 옳은 것은 데이터입니다.** 발제문 참고사항에는 `price` 가 **"SEK(스웨덴 크로나)"** 이고 기간이 **"수년간"** 이라고 적혀 있습니다. 직접 확인해 보세요 — `df['price'].min()`·`max()` 와 `df['t_dat'].min()`·`max()` 를 찍어 보면, `price` 는 0 과 1 사이의 **정규화된 상대값**이고 기간은 **1년**입니다. 그래서 이 프로젝트는 price 를 금액이 아니라 **상대값**으로만 다루고("몇 크로나"가 아니라 "어느 쪽이 몇 배"), **"연도별 추세"는 이 데이터로 말하지 않습니다.** 남이 써 준 설명을 데이터로 검증하는 것도 분석가의 일입니다.

**파생변수 만들기 — 이후 절에서 쓸 `month`·`weekday`·`age_group`**
정제본에 이미 있어도 다시 계산해 둡니다(재현 경로로 왔을 때도 항상 있도록). 이후 EDA·기술통계에서 계속 쓸 파생변수 세 개입니다(day07 파생변수 레시피 — 날짜 분해·구간화). 이 노트북의 나머지 절은 전부 이 파생변수가 있다고 가정합니다.

**1-3. 파생변수 3개와 표본 1개**

- **무엇을 만드나**: `df` 에 컬럼 세 개를 **이 이름 그대로** 추가하고, 무거운 그림·집계에 재사용할 표본 `df_s` 를 만듭니다. `month` = 거래일의 월(1~12 정수), `weekday` = 거래일의 요일 이름(Monday…), `age_group` = 나이를 10살 단위로 내림한 정수(23세 → 20), `df_s` = 3,000행 무작위 표본.
- **왜**: **이 노트북의 뒤 절과 03_데이터_통계가 이 컬럼 이름을 그대로 씁니다.** `달`·`요일`·`나이대` 처럼 다른 이름으로 만들면 뒤의 지침과 03 이 그 컬럼을 찾지 못합니다.
- **어떤 도구를 쓰나**:
  - 월·요일: 날짜형 컬럼의 `.dt` 접근자 — `.dt.month`, `.dt.day_name()`.
  - 연령대: 정수 나눗셈으로 10 단위로 내린 뒤(`나이 // 10 * 10` 꼴) `.astype(int)` 로 정수화.
  - 표본: `df.sample(n=…, random_state=42)` — `random_state` 를 고정해야 매번 같은 표본이 나와 결과를 비교할 수 있습니다.
- **이렇게 나오면 맞다**: `df[['month', 'weekday', 'age_group']].head()` 에 1~12 정수·영어 요일 이름·10 단위 정수가 보이고, `len(df_s)` 가 3000 입니다.
- **함정**: `.dt` 는 **날짜형 컬럼에만** 붙습니다(1-2 에서 변환했는지 확인). `age_group` 을 정수로 바꾸지 않으면 20.0 처럼 소수로 찍혀 그래프 눈금이 지저분해집니다.

In [ ]:
# 여기에 코드를 작성하세요

**짚고 넘어가기 — merge 후 행이 늘지 않았는지 검증**
01 에서 거래·고객·상품을 merge 할 때 다대다(many-to-many) 매칭이 있으면 행이 **예상보다 훨씬 많이** 늘어납니다(뻥튀기). 지금 정제본을 받았어도, 습관적으로 **거래를 유일하게 식별하는 조합**(고객·상품·날짜)에 중복이 없는지 한 번 더 확인하세요.

**1-4. 중복 검증**

- **무엇을 만드나**: `customer_id`·`article_id`·`t_dat` 세 컬럼을 묶었을 때 중복이 몇 건인지 세어, 전체 행수와 함께 한 줄로 출력합니다.
- **어떤 도구를 쓰나**:
  - 중복 여부: `df.duplicated(subset=[<컬럼 목록>])` — 행마다 True/False 를 돌려줍니다.
  - 건수: 그 결과에 `.sum()` (True 가 1 로 더해집니다).
  - 전체 행수: `len(df)`.
- **이렇게 나오면 맞다**: "중복 N건 / 전체 M행" 이 찍히고, N 이 전체(11만여 행)에 비해 **수십 건 이하**로 아주 작습니다.
- **함정**: `subset` 을 빼면 **모든 컬럼이 같은 행**만 세므로 거래 중복을 못 잡습니다. `.sum()` 대신 `.count()` 를 쓰면 True 개수가 아니라 전체 행수가 나옵니다.

In [ ]:
# 여기에 코드를 작성하세요

> ⚠️ 0 이 아니어도 바로 오류는 아닙니다 — 같은 고객이 같은 날 같은 상품을 **여러 번** 사면 똑같은 조합이 나올 수 있습니다(수량 컬럼이 없을 때). 이 정제본은 그 비율이 전체 대비 매우 작습니다 — **비율이 크다면** merge 방식(다대다 매칭 여부)을 의심하세요.

> ✅ **여기까지 되면 통과**: `df.shape` 가 출력되고 위 `head()` 표가 정상적으로 보이면 됩니다(정확한 행수는 01 에서 고른 규칙에 따라 달라질 수 있습니다).

## 2. 가이드 — 막힐 때 여기를 보세요
**이 절에서 할 일**: 없습니다 — 이 절은 **어떤 도구를 쓸지 고르는 지도**입니다. 순서대로 읽지 말고, 3·4절을 하다 막히면 필요한 부분만 찾아보세요.

**여기에도 완성 코드는 없습니다.** 각 항목은 "무엇을 만드나 → 어떤 도구(함수·핵심 인자) → 이렇게 나오면 맞다 → 함정" 으로 되어 있습니다. 도구 이름을 알았으면 **직접 써 보고**, 인자가 헷갈릴 때는 `sns.histplot?` 처럼 물음표를 붙여 실행하면 그 함수의 설명이 나옵니다.

### 2-1. 집계 도구 선택 기준 — `groupby` vs `pivot_table` vs `crosstab`
셋 다 "그룹으로 묶어 본다"는 같은 아이디어지만, **무엇을 보고 싶은지**에 따라 고릅니다.

| 하고 싶은 것 | 도구 |
|---|---|
| 그룹별로 **여러 통계**(평균·합계·개수 등)를 한 표로 | `groupby` + `agg` |
| **두 범주형** 변수를 행·열로 펼쳐 **수치형 값**(주로 평균·합계)을 비교 | `pivot_table` |
| **두 범주형** 변수의 **빈도(건수)** 를 표로, 필요하면 비율까지 | `crosstab` |

**집계 도구 다섯 가지**

- **무엇을 만드나**: 제품군별 요약표(건수·평균가·총매출), 연령대 × 채널 평균가 표, 멤버십 × 채널 비율표, 상품별 매출 상위 10, 채널별 비중 — 이 다섯 가지를 각각 만들어 보세요.
- **어떤 도구를 쓰나**:
  - `df.groupby(<키>, observed=True).agg(<새 컬럼 이름>=(<대상 컬럼>, <함수 이름>), …)` — 함수 이름은 `'count'`·`'mean'`·`'sum'` 처럼 문자열로.
  - `pd.pivot_table(df, index=…, columns=…, values=…, aggfunc='mean')`
  - `pd.crosstab(<행 시리즈>, <열 시리즈>)` — 비율로 보려면 `normalize='index'`(행 기준) 또는 `'columns'`·`'all'`.
  - 상위 N: `<시리즈>.nlargest(<n>)` (= `sort_values(ascending=False).head(n)`).
  - 비중: `<시리즈>.value_counts(normalize=True)`.
  - 정렬 `.sort_values(<컬럼>, ascending=False)`, 자릿수 `.round(<자리>)`, 표 출력 `display(...)`.
- **이렇게 나오면 맞다**: 집계표의 행이 원하는 순서로 정렬돼 보이고, `normalize='index'` 로 만든 비율표는 **각 행의 합이 1** 입니다.
- **함정**: `observed=True` 를 빼면 pandas 버전에 따라 **한 건도 없는 범주 조합까지** 표에 나옵니다. `agg` 의 `(컬럼, 함수)` 순서를 뒤집으면 에러입니다. groupby 결과의 키는 컬럼이 아니라 인덱스로 들어갑니다(2-5 참고).

### 2-2. 커스텀 집계 — `lambda`·사용자 함수
평균·합계·개수로 안 되는 질문("범위가 얼마나 되나", "상위 10% 비율은?", "가장 흔한 가격대는?")은 `agg` 에 **함수를 직접 넣어** 구합니다.

**커스텀 집계 — 연령대별 지표 3개**

- **무엇을 만드나**: 연령대별로 ① price 의 범위(최대−최소) ② 전체 상위 10% 기준을 넘는 거래의 비율 ③ 소수 2자리로 반올림한 최빈 가격대 — 세 지표를 한 표로 만듭니다.
- **어떤 도구를 쓰나**:
  - `agg(<새 컬럼 이름>=lambda s: …)` — `s` 는 그룹별 시리즈 하나입니다.
  - 범위는 `s.max() - s.min()`, 분위 기준은 `s.quantile(0.9)`.
  - 비율은 **조건식의 평균** — `(<조건>).mean()` 이 True 의 비율입니다.
  - 최빈값은 `.mode()` (여러 값이 나올 수 있어 하나만 쓸 때는 `.iloc[0]` 로 첫 값을 꺼냅니다).
- **이렇게 나오면 맞다**: 연령대마다 세 컬럼이 서로 다른 값으로 채워지고, 고가비율이 대략 0.1 안팎에서 연령대별로 조금씩 다릅니다.
- **함정**: `lambda` 안의 `s` 는 **그 그룹만의 시리즈**입니다 — 전체 기준으로 비교하고 싶다면 `df['price'].quantile(0.9)` 처럼 전체에서 계산한 값을 써야 합니다(무엇을 기준으로 삼을지 먼저 정하세요).

### 2-3. 시각화 갤러리 — 변수 개수·유형으로 고르기
그래프를 고를 때 헤매지 않는 방법은 **"몇 개의 변수를, 어떤 유형으로 볼 것인가"** 를 먼저 정하는 것입니다. 무거운 그림은 전체 대신 **3,000행 표본**(1절에서 만든 `df_s`)으로 그립니다 — 경향은 같고 훨씬 빠릅니다.

| 변수 개수 | 유형 | 그래프 |
|---|---|---|
| **단변량** | 수치 | `histplot`·`boxplot`(+ 로그축) |
| **단변량** | 범주 | `countplot` |
| **이변량** | 수치 × 수치 | `scatterplot`·`regplot` |
| **이변량** | 범주 × 수치 | `barplot`·`boxplot`(+ `hue`) |
| **이변량** | 범주 × 범주 | `crosstab` + `heatmap` |
| **다변량** | 셋 이상 | `hue`·`subplots`(소형 다중 그림)·상관 `heatmap` |
| **시간** | 추세 | `lineplot`·롤링 평균 |

**단변량 — 수치: `histplot`·`boxplot`**
**언제**: 한 수치 변수가 **어떻게 퍼져 있는지**(치우침·봉우리 개수) 볼 때 → `histplot`. **중앙값·사분위·이상치**를 한눈에 볼 때 → `boxplot`.

**분포 그림 — 히스토그램 2개(원래 축·로그축)와 연령대별 상자그림**

- **무엇을 만드나**: price 의 히스토그램을 원래 축과 로그축으로 **나란히** 그려 비교하고, 이어서 연령대별 price 상자그림을 그립니다.
- **어떤 도구를 쓰나**:
  - 나란히 두 칸: `fig, axes = plt.subplots(1, 2, figsize=…)` — 각 그림에 `ax=axes[0]`·`ax=axes[1]` 로 어느 칸에 그릴지 지정합니다.
  - `sns.histplot(<시리즈>, bins=…, log_scale=True, ax=…)` — 로그축은 `log_scale=True` 한 줄로.
  - `sns.boxplot(data=…, x=<범주 컬럼>, y=<수치 컬럼>)`
  - 제목은 칸별로 `<ax>.set_title(…)`, 그림 하나면 `plt.title(…)`. 마무리는 `plt.tight_layout()` 과 `plt.show()`.
- **이렇게 나오면 맞다**: 원래 축에서는 값이 왼쪽에 몰린 **오른쪽 꼬리**가 보이고, 로그축에서는 봉우리가 가운데로 와서 좌우 대칭에 가까워집니다. 상자그림에서는 상자 위쪽으로 점(이상치)이 여럿 보입니다.
- **함정**: 무거운 그림은 `df` 전체가 아니라 `df_s` 로. 한 셀에서 그림을 여러 개 그릴 때는 **그리기 전에 `plt.figure()` 나 `plt.subplots()` 로 새 그림판을 만들어야** 앞 그림 위에 겹쳐 그려지지 않습니다.

**단변량 — 범주: `countplot`**
**언제**: 범주별 **건수(빈도)** 만 볼 때 — 집계 없이 원본을 바로 넣습니다.

**범주 빈도 막대 — 채널별 거래 건수**

- **무엇을 만드나**: 채널(`sales_channel_id`)별 거래 건수를 막대로 그립니다 — **미리 집계하지 않고** 원본 컬럼을 그대로 넣습니다.
- **어떤 도구를 쓰나**:
  - `sns.countplot(data=…, x=<범주 컬럼>)` — 건수는 함수가 스스로 셉니다.
  - 그림판은 `plt.figure(figsize=…)`, 제목은 `plt.title(…)`.
- **이렇게 나오면 맞다**: 막대 두 개(채널 1·2)가 나오고 채널 2 가 더 높습니다. 다만 **이 격차를 그대로 "온라인이 강하다"로 읽지 마세요** — 이 데이터는 Kaggle 에 공개된 표본이고, 발제문도 **온라인(채널 2) 기록이 상대적으로 많이 담겼을 수 있다**고 경고합니다. 건수 비중은 **표본이 어떻게 모였는지**에 따라 달라지므로, 채널을 비교하려면 건수보다 **비율(%)이나 고객당 평균 구매액(ARPU)** 이 안전합니다(선택 미션에서 다룹니다).
- **함정**: `value_counts()` 로 이미 센 결과를 `countplot` 에 넣으면 **이중 집계**가 됩니다 — 집계된 값을 그릴 때는 `barplot` 을 쓰세요.

**이변량 — 수치 x 수치: `scatterplot`·`regplot`**
**언제**: 두 수치 변수의 **관계**를 점으로 볼 때 → `scatterplot`(`hue` 로 세 번째 변수도). **추세선까지** 보고 싶을 때 → `regplot`.

**관계 그림 — 산점도와 추세선**

- **무엇을 만드나**: 나이(`age`)와 price 의 산점도를 채널별 색으로 그리고, 옆 칸에 추세선이 있는 그림을 그려 비교합니다.
- **어떤 도구를 쓰나**:
  - `sns.scatterplot(data=…, x=…, y=…, hue=<색으로 나눌 컬럼>, alpha=<투명도>, ax=…)`
  - `sns.regplot(data=…, x=…, y=…, scatter_kws={'alpha': 0.3}, ax=…)` — 점의 투명도는 `scatter_kws` 로 따로 넘깁니다.
- **이렇게 나오면 맞다**: 점이 넓게 흩어져 뚜렷한 직선이 보이지 않고, 추세선이 거의 평평합니다(관계가 약하다는 뜻).
- **함정**: 점이 수만 개면 서로 겹쳐 형태가 안 보입니다(오버플로팅) — 표본 `df_s` 를 쓰고 `alpha` 로 투명하게. `regplot` 은 `hue` 를 받지 않습니다(그룹별 추세선이 필요하면 그룹마다 따로).

**이변량 — 범주 x 수치: `barplot`·`boxplot`(+ `hue`)**
**언제**: 범주별 **평균 같은 대표값**을 비교할 때 → `barplot`(신뢰구간까지 자동). 범주별 **분포 전체**를 비교하고, 거기에 **세 번째 범주까지** 겹쳐 보고 싶을 때 → `boxplot(hue=...)`.

**범주별 비교 그림 — 정렬된 막대와 `hue` 상자그림**

- **무엇을 만드나**: ① 제품군별 평균가 막대를 **평균이 높은 순서로 정렬**해 그리고, ② 채널별 price 분포를 멤버십 상태(`hue`)로 한 번 더 나눈 상자그림을 그립니다.
- **어떤 도구를 쓰나**:
  - 정렬 순서: 평균을 구해 내림차순 정렬한 결과의 `.index` 를 `sns.barplot(..., order=<그 순서>)` 에 넘깁니다.
  - `sns.boxplot(data=…, x=…, y=…, hue=<세 번째 범주>)`
  - x 라벨이 겹치면 `plt.xticks(rotation=45, ha='right')`, 범례가 그림을 가리면 `plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')`.
- **이렇게 나오면 맞다**: 막대가 왼쪽에서 오른쪽으로 낮아지고, x 축 라벨이 겹치지 않으며 범례가 그림을 가리지 않습니다.
- **함정**: `order` 를 안 주면 막대가 **데이터에 나온 순서**대로 뒤죽박죽 놓입니다. `hue` 범주가 많으면 상자가 너무 얇아져 읽을 수 없습니다(범주를 묶거나 개수를 줄이세요).

**이변량 — 범주 x 범주: `crosstab` + `heatmap`**
**언제**: 두 범주형 변수의 조합별 **건수**를 색으로 한눈에 볼 때.

**교차 히트맵 — 연령대 × 채널 건수**

- **무엇을 만드나**: 연령대와 채널의 조합별 거래 건수를 교차표로 만들고, 그 표를 색으로 칠한 히트맵을 그립니다.
- **어떤 도구를 쓰나**:
  - 표: `pd.crosstab(<행 시리즈>, <열 시리즈>)`.
  - 그림: `sns.heatmap(<표>, annot=True, fmt='d', cmap='Blues')` — `annot=True` 면 칸마다 숫자를 찍고, `fmt` 는 그 숫자 형식입니다.
- **이렇게 나오면 맞다**: 칸마다 정수 건수가 찍히고, 20~30대 × 채널 2 칸이 가장 진합니다.
- **함정**: `fmt='d'` 는 **정수용**입니다 — `normalize=` 로 비율표를 만들었다면 `fmt='.2f'` 로 바꾸지 않으면 포맷 에러가 납니다.

**다변량 — `hue`·`subplots`·상관 `heatmap`**
**언제**: "전체 그림이 그룹마다 다르게 보이는가?" 를 확인할 때. 그룹이 **2~3개**면 `hue` 로 한 그림에 겹쳐 보고, **여러 개**면 `subplots` 로 나란히(소형 다중 그림). **여러 수치 변수 사이 관계를 한 번에** 보려면 상관 `heatmap`.

**다변량 그림 3종 — 겹쳐 보기 · 나란히 보기 · 상관 히트맵**

- **무엇을 만드나**: ① 채널별 price 분포를 한 그림에 겹쳐 그리고, ② 상품 대분류(`index_group_name`)마다 히스토그램을 나란히 그리고, ③ price·age·month 세 수치 변수의 상관 히트맵을 그립니다.
- **어떤 도구를 쓰나**:
  - 겹쳐 보기: `sns.histplot(data=…, x=…, hue=…, element='step', log_scale=True)` — `element='step'` 이면 막대가 아니라 계단선이라 겹쳐도 읽힙니다.
  - 나란히 보기: 그룹 목록은 `<시리즈>.unique()`, 칸은 `plt.subplots(1, <칸 수>, figsize=…, sharey=True)`, 반복은 `for ax, g in zip(axes, <그룹 목록>)`, 그룹별 데이터는 `df_s.loc[<조건>, <컬럼>]`.
  - 상관 히트맵: `<여러 컬럼 DataFrame>.corr()` 로 상관행렬을 만들고 `sns.heatmap(..., annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)`.
- **이렇게 나오면 맞다**: 겹친 그림에서 두 채널의 봉우리 위치가 비슷하고, 히트맵의 대각선이 1.00 이며 나머지 칸은 0 에 가깝습니다.
- **함정**: `subplots` 의 칸이 1개면 `axes` 가 배열이 아니라 단일 객체여서 `zip` 이 돌지 않습니다. 상관 히트맵은 `vmin=-1, vmax=1` 을 고정해야 값이 작을 때 색이 과장되지 않습니다.

**시간 — `lineplot`·롤링 평균**
**언제**: 시간(월·일 등)에 따른 **변화 추이**를 볼 때. `lineplot` 은 x 값이 같은 여러 점을 자동으로 평균 내 그려줍니다. 하루하루가 들쭉날쭉하면 **롤링(이동) 평균**으로 큰 흐름만 봅니다.

**시간 추세 그림 — 월별 꺾은선과 일별 + 7일 롤링 평균**

- **무엇을 만드나**: ① 월별 평균가를 꺾은선으로 그리고, ② 일별 평균가 위에 **7일 롤링 평균**을 겹쳐 그려 노이즈가 얼마나 걷히는지 봅니다.
- **어떤 도구를 쓰나**:
  - 집계 후 그리기: 집계 결과에 `reset_index()` 를 붙여 그룹 키를 다시 컬럼으로 만든 뒤 `sns.lineplot(data=…, x=…, y=…, marker='o')`.
  - x 눈금을 1~12 로 고정: `plt.xticks(range(1, 13))`.
  - 일 단위 집계: `df.groupby(<날짜 시리즈>)` — 날짜만 뽑을 때는 `.dt.date`. 시리즈를 프레임으로 돌릴 때 값 컬럼 이름은 `reset_index(name=<이름>)` 로 지정합니다.
  - 롤링 평균: `<시리즈>.rolling(<창 크기>).mean()`.
  - 두 선 겹치기: `plt.plot(<x>, <y>, label=…)` 를 두 번 부르고 `plt.legend()`.
- **이렇게 나오면 맞다**: 롤링 선이 원래 선보다 훨씬 매끄럽고, 창 크기보다 앞선 며칠은 값이 없어(NaN) 선이 조금 늦게 시작합니다.
- **함정**: 롤링 전에 **날짜순 정렬**(`sort_values`)이 안 되어 있으면 이동 평균이 뒤죽박죽 됩니다. `groupby(.dt.date)` 의 결과 인덱스는 날짜형이 아니라 `date` 객체라, x 축을 날짜로 쓰려면 `pd.to_datetime` 으로 다시 바꿔야 합니다.

### 2-4. 기술통계 도구 (day08)
**추론통계(검정)는 03 담당**입니다 — 여기서는 데이터를 **요약해서 설명**하는 도구만 다룹니다.

**대표값 — 평균·중앙값·최빈값·절사평균·가중평균**
**평균의 함정**: 평균은 **극단값에 약합니다**. `price` 처럼 소수의 아주 비싼 거래가 있으면 평균이 중앙값보다 위로 끌려 올라갑니다 — 이럴 땐 **중앙값**이나, 양 끝 극단치를 잘라내는 **절사평균**(`stats.trim_mean`)이 더 안정적입니다.

**대표값 다섯 가지**

- **무엇을 만드나**: price 의 평균·중앙값·최빈값·절사평균(위아래 10% 잘라낸 평균)·가중평균(나이로 가중)을 구해 **소수 4자리로 한 줄에** 출력하고, 그 아래에 무엇을 알 수 있는지 한 줄로 적습니다.
- **어떤 도구를 쓰나**:
  - `.mean()`·`.median()`·`.mode()` (최빈값은 여러 개일 수 있어 `.iloc[0]` 로 첫 값).
  - 절사평균: `stats.trim_mean(<값>, <잘라낼 비율>)` — 0.1 이면 위·아래 10% 씩.
  - 가중평균: `np.average(<값>, weights=<가중치>)`.
  - 출력 형식은 f-string 의 `:.4f` 나 `%` 포맷으로 자릿수를 맞춥니다.
- **이렇게 나오면 맞다**: **평균 > 중앙값** 입니다(오른쪽 꼬리 분포의 신호). 절사평균은 평균보다 중앙값 쪽에 가깝습니다.
- **함정**: `.mode()` 는 시리즈를 돌려주므로 그냥 쓰면 표가 됩니다. 가중평균의 가중치에 결측이 있으면 결과가 NaN 이 됩니다.

**산포 — 범위·IQR·분산/표준편차·변동계수(CV)**
표준편차는 **단위가 있어서** 서로 다른 변수끼리 "어느 쪽이 더 퍼져 있나" 비교하기 어렵습니다. 이럴 때 **변동계수 CV = 표준편차 / 평균**(단위 없음)으로 비교합니다.

**산포 네 가지 + 변수 간 비교**

- **무엇을 만드나**: price 의 범위·IQR·표준편차(표본 기준)·CV 를 구하고, **age 의 CV 도 함께** 출력해 어느 변수가 상대적으로 더 퍼져 있는지 비교합니다.
- **어떤 도구를 쓰나**:
  - 범위: `.max() - .min()`.
  - IQR: `.quantile([0.25, 0.75])` 로 Q1·Q3 를 한 번에 받아 그 차이를 구합니다.
  - 표준편차: `.std(ddof=1)` (표본 기준).
  - CV: 표준편차를 평균으로 나눈 값 — 단위가 없어서 다른 변수와 비교할 수 있습니다.
- **이렇게 나오면 맞다**: price 의 CV 가 age 의 CV 보다 **큽니다** — 단위가 다른 두 변수를 표준편차만으로는 비교할 수 없다는 것을 확인하게 됩니다.
- **함정**: `ddof` 기본값이 **pandas 는 1(표본), numpy 는 0(모집단)** 으로 다릅니다 — 표본 기준이면 `ddof=1` 을 명시하세요.

**분포 형태 — 왜도·첨도, 로그 변환**
**왜도(skewness)**: 0 이면 좌우 대칭, 양수면 오른쪽 꼬리(고가 이상치)가 깁니다. **첨도(kurtosis)**: 0(정규분포 기준)보다 크면 꼬리가 두꺼워 **극단값이 잦다**는 뜻입니다 — "꼬리 리스크". 오른쪽으로 심하게 치우친 변수는 **로그 변환**으로 대칭에 가깝게 만들 수 있습니다.

**분포 형태 — 왜도·첨도와 로그 변환 전후 비교**

- **무엇을 만드나**: price 의 왜도·첨도, 그리고 **로그 변환 후**의 왜도·첨도까지 네 값을 한 줄로 출력하고, 변환 전후 히스토그램을 나란히 그려 눈으로도 확인합니다.
- **어떤 도구를 쓰나**:
  - `stats.skew(<값>)`·`stats.kurtosis(<값>)`.
  - 로그 변환: `np.log(<값>)`.
  - 그림은 2-3 절의 `subplots` + `histplot` 을 그대로 활용하고, 제목에 왜도 값을 f-string 으로 넣으면 비교가 쉽습니다.
- **이렇게 나오면 맞다**: 원본 왜도는 2 를 넘고(오른쪽 꼬리), 로그 변환 후에는 0 근처로 내려옵니다. 그림에서도 봉우리가 가운데로 옮겨옵니다.
- **함정**: `np.log` 는 0 이하 값에서 `-inf`·NaN 을 냅니다(01 에서 0 이하를 걸렀는지 확인 — 0 이 섞일 수 있으면 `np.log1p`). `stats.kurtosis` 는 기본이 **초과첨도**라 정규분포가 0 입니다(3 이 아닙니다).

**상관 — 피어슨 vs 스피어만, 상관 ≠ 인과**
**피어슨**: 두 변수의 **선형** 관계 강도(직선에 얼마나 가까운지). **스피어만**: 값 대신 **순위**로 계산해 **단조(monotonic) 관계**(반드시 직선이 아니어도 한쪽이 늘 때 다른 쪽도 느는지)를 봅니다 — 이상치에 덜 민감합니다. **둘의 차이가 크면** 관계가 있어도 **비선형**이라는 신호입니다.

> ⚠️ **상관은 인과가 아닙니다.** 두 변수가 함께 움직인다고 한쪽이 다른 쪽의 **원인**이라는 뜻은 아닙니다 — 제3의 변수(공통 원인)나 우연일 수 있습니다.

**상관 — 피어슨과 스피어만 비교**

- **무엇을 만드나**: price 와 age 의 피어슨 상관계수와 스피어만 상관계수를 각각 구해 한 줄로 나란히 출력하고, 두 값의 차이가 무엇을 뜻하는지 한 줄로 적습니다.
- **어떤 도구를 쓰나**:
  - `df[[<컬럼1>, <컬럼2>]].corr(method='pearson')` / `method='spearman'`.
  - 결과는 2×2 행렬이므로 원하는 칸을 `.iloc[0, 1]` 로 꺼냅니다.
  - 여러 변수를 한 번에 보려면 2-3 절의 상관 히트맵을 그대로 씁니다.
- **이렇게 나오면 맞다**: 두 값이 모두 0 에 가깝고 서로 비슷합니다 → 뚜렷한 비선형 패턴 없이 **관계 자체가 약하다**고 읽습니다.
- **함정**: `.corr()` 결과를 그대로 출력하면 행렬이 나옵니다(칸을 꺼내세요). 그리고 상관이 크게 나와도 **인과가 아닙니다** — 위 ⚠️ 를 다시 읽으세요.

### 2-5. 자주 나는 에러와 해결
이 파트에서 자주 나는 에러 세 가지입니다. **증상 → 원인 → 어떻게 고치나** 순서로 되어 있으니, 막혔을 때 여기로 돌아와 증상부터 찾으세요.

**에러 ① `SettingWithCopyWarning` — 걸러낸 결과에 새 컬럼을 넣을 때**

- **증상**: `df[<조건>]` 로 걸러낸 결과에 새 컬럼을 넣으면 `SettingWithCopyWarning: A value is trying to be set on a copy of a slice from a DataFrame` 가 뜬다.
- **원인**: 걸러낸 결과가 원본의 **사본인지 원본을 가리키는 창(view)인지** pandas 가 확신할 수 없어서, 그 대입이 원본에 반영될지 보장하지 못한다고 경고하는 것이다.
- **어떻게 고치나**: 필터링 결과를 계속 수정할 거라면 뒤에 **`.copy()`** 를 붙여 원본과 독립된 사본으로 만든다. 확인은 새 컬럼 이름이 사본의 `.columns` 에는 있고 원본 `df` 에는 없는지 보면 된다.

**에러 ② 그래프의 한글이 네모(□)로 보인다**

- **증상**: 그래프의 제목·축 라벨·범례에서 한글이 네모(□)로 보이고, 실행할 때 `Glyph … missing from font` 경고가 뜬다.
- **원인**: 지금 쓰이는 폰트에 한글 글자가 없다. 1절에서 폰트를 지정했더라도 그 뒤에 `sns.set_theme()` 을 부르면 폰트 설정이 되돌아간다.
- **어떻게 고치나**: 먼저 `plt.rcParams['font.family']` 를 출력해 **지금 어떤 폰트인지** 확인한다. OS 에 맞는 폰트 이름이 아니면 1절 준비 셀을 다시 실행하고, `set_theme` 을 쓸 때는 `font=` 로 폰트를 함께 넘긴다(리눅스·Colab 은 나눔 폰트가 설치돼 있어야 한다).

**에러 ③ `groupby` 뒤에 그룹 키 컬럼이 사라졌다**

- **증상**: `groupby` 집계 결과를 seaborn 에 넣으니 `x='age_group'` 에서 그런 컬럼이 없다는 에러가 나고, 결과의 `.columns` 목록에도 그룹 키가 없다.
- **원인**: 그룹 키는 컬럼이 아니라 **인덱스**로 들어가기 때문이다(집계 결과가 시리즈면 컬럼 이름 자체가 없다).
- **어떻게 고치나**: `reset_index()` 로 인덱스를 다시 일반 컬럼으로 되돌린다. 집계 결과가 시리즈일 때는 `reset_index(name=<값 컬럼 이름>)` 로 값 컬럼의 이름까지 정해 주면 그래프에 바로 넣을 수 있다. 확인은 `.columns` 목록에 그룹 키가 보이는지.

## 3. 필수 EDA 미션
> ⚠️ **이 절은 필수입니다.** 발제문의 필수 분석 목표 중 이 파트가 맡은 두 가지입니다. **자가채점은 없습니다** — 체크포인트로 스스로 확인하세요.

### 필수 미션 1 — 기술통계 제시
정제본의 **컬럼 타입·결측**과, 주요 수치 컬럼(`price`·`age` 등)의 **대표값**(평균·중앙값 등)과 **산포**(표준편차 등)를 표로 제시하세요. 2-4 절의 도구를 활용해도 좋습니다.

> 🔧 **막히면 볼 곳**: 2-4 기술통계 도구(대표값·산포).

> ✅ **여기까지 되면 통과**: 수치형은 min/median/mean/max, 범주형은 value_counts 상위 몇 개가 나오면 됩니다 — 형식은 자유입니다.

In [ ]:
# 여기에 코드를 작성하세요

### 필수 미션 2 — 기준 컬럼 비교분석 + 시각화
**기준 컬럼을 하나 이상 골라**(연령대·채널·상품군 중, 또는 다른 컬럼도 좋습니다) **집계함수로 비교분석**하고, **최소 3개 이상의 시각화**로 보여주세요(2-3 절의 변수 유형 표를 참고해 단변량·이변량을 섞어도 좋습니다). 그림마다 **무엇을 말하는지 한 줄**을 답니다.

> 🔧 **막히면 볼 곳**: 2-1 집계 도구 선택 기준, 2-3 시각화 갤러리.

> ✅ **여기까지 되면 통과**: 고른 기준 컬럼으로 최소 하나의 groupby/pivot_table 집계 표 + 3개 이상 그림 + 각 그림 아래 한 줄 해석이 있으면 됩니다.

In [ ]:
# 여기에 코드를 작성하세요

## 4. 선택 EDA 미션 카탈로그
> 🔧 **이 절은 선택입니다.** 원하는 것만 골라서 하세요 — 전부 할 필요 없습니다. 각 미션은 **발제문의 심화 분석 목표**를 문제로 만든 것이고, 2절의 도구 지도를 응용하면 대부분 시작할 수 있습니다.

| 난이도 | 선택 미션 | 쓰는 도구 | 예상 시간 |
|---|---|---|---|
| ★ | 1. 산점도 등 다양한 형식으로 시각화 | `scatterplot`·`pairplot` | 15분 |
| ★ | 2. 고객 속성 × 채널 교차분석 | `crosstab`·`heatmap` | 15분 |
| ★★ | 3. 로그 변환·정규화(z-score·Min-Max)로 분포 비교 | `np.log`·표준화 계산 | 20분 |
| ★★ | 4. 이상치 고객·거래 탐색 전략 | IQR 규칙·`boxplot` | 20분 |
| ★★ | 5. 월별·분기별 평균 매출 — 추세와 시즌성 | `dt.quarter`·`lineplot` | 20분 |
| ★★ | 6. 채널 비중 해석 보정 — 비율(%)과 ARPU | `nunique`·비율 계산 | 20분 |
| ★★★ | 7. 고객별 총 구매액 상위 20% vs 하위 20% | `qcut`·`groupby` | 25분 |
| ★★★ | 8. 월별 매출의 3개월 롤링 평균 | `rolling().mean()` | 20분 |

> ⚠️ **이 데이터의 기간은 1년(2019-01-01 ~ 2019-12-31)입니다.** 그래서 5·8번은 **"연도별 추세"가 아니라 "1년 안의 월별 흐름"** 까지만 말할 수 있습니다. 발제문에는 "수년간"이라고 적혀 있지만, 여러분의 데이터로 직접 확인한 사실이 우선입니다.

### 선택 미션 1 (★) — 산점도 등 다양한 형식으로 시각화
> 발제문 심화 분석 목표 ①

**비즈니스 질문**: "많이 사는 고객"과 "비싸게 사는 고객"은 같은 사람인가?

- **무엇을 만드나**: **고객 한 명을 점 하나로** 만든 표(구매건수·총구매액·평균구매액)를 준비해 산점도를 그리고, 세 변수를 한 번에 보는 격자 그림(`pairplot`)까지 그립니다. 거래 한 건이 아니라 **고객 한 명**이 관측 단위가 되는 것이 핵심입니다.
- **어떤 도구를 쓰나**:
  - 고객 단위 표: `df.groupby('customer_id').agg(…)` 로 건수·합계를 구하고, 평균구매액은 두 컬럼을 나눠 만듭니다.
  - 산점도: `sns.scatterplot(data=…, x=…, y=…, alpha=…)` — 점이 많으면 `<표>.sample(n=…, random_state=42)` 로 표본을 씁니다.
  - 여러 수치 변수를 한 번에: `sns.pairplot(<수치 컬럼만 남긴 표>, height=…)` — 대각선은 각 변수의 분포, 나머지 칸은 두 변수의 산점도입니다.
  - 관계의 세기를 숫자로도 보고 싶으면 2-4 절의 상관 도구.
- **이렇게 나오면 맞다**: 산점도에서 구매건수가 늘면 총구매액도 대체로 늘지만(우상향), **평균구매액과 구매건수 사이에는 그런 경향이 거의 안 보입니다** — 이 두 그림의 차이를 한 줄로 설명할 수 있으면 성공입니다.
- **함정**: 고객 대부분이 구매 1~2건이라 점이 왼쪽에 뭉칩니다 — `alpha` 로 투명하게 하거나 표본을 쓰세요. `pairplot` 은 그림 단위 함수라 `figsize` 대신 `height` 를 씁니다(수만 행을 그대로 넣으면 매우 느립니다).

In [ ]:
# 여기에 코드를 작성하세요

### 선택 미션 2 (★) — 고객 속성 × 채널 교차분석
> 발제문 심화 분석 목표 ⑦

**비즈니스 질문**: 멤버십 상태나 뉴스 구독 습관에 따라 **주로 쓰는 채널**이 다른가?

- **무엇을 만드나**: 고객 속성(`club_member_status`·`fashion_news_frequency`·`age_group` 중 둘 이상)과 채널의 교차표를 만들고, **행 기준 비율**로 바꿔 히트맵으로 보여줍니다. 비율표와 함께 **건수표도** 제시해 표본이 작은 범주를 드러냅니다.
- **어떤 도구를 쓰나**:
  - 교차표: `pd.crosstab(<속성 시리즈>, <채널 시리즈>)` — 비율은 `normalize='index'`.
  - 히트맵: `sns.heatmap(<비율표>, annot=True, fmt='.2f', cmap='Blues')`.
  - 표본 크기 확인: 비율표와 나란히 `normalize` 없는 건수표를 함께 출력합니다.
- **이렇게 나오면 맞다**: 비율표와 건수표를 나란히 놓았을 때, **비중이 극단적으로 보이는 범주가 실은 건수가 아주 적다**는 것을 스스로 지적할 수 있으면 성공입니다. 어떤 속성에서는 범주 간 비중 차이가 거의 없을 수도 있는데, 그 "차이가 없다"도 결론입니다.
- **함정**: 비율만 보면 **50건짜리 범주와 10만 건짜리 범주가 똑같은 크기의 칸**으로 보입니다 — 건수를 함께 보지 않으면 "탈퇴 고객은 96%가 온라인" 같은 문장을 근거 없이 쓰게 됩니다. `fmt='d'` 는 비율표에서 에러가 납니다(`'.2f'` 로).

In [ ]:
# 여기에 코드를 작성하세요

### 선택 미션 3 (★★) — 로그 변환·정규화(z-score·Min-Max)로 분포 비교
> 발제문 심화 분석 목표 ④

**비즈니스 질문**: 치우친 가격 분포를 **어떻게 손봐야** 고객·상품군끼리 공정하게 비교할 수 있나?

- **무엇을 만드나**: price 에 ① 로그 변환 ② z-score 표준화 ③ Min-Max 정규화를 각각 적용해, **네 가지 버전(원본 포함)의 왜도와 히스토그램**을 비교합니다. 그다음 z-score 를 써서 **상품 대분류별 평균 z** 를 구해 어느 군이 전체 평균보다 비싼지 봅니다.
- **어떤 도구를 쓰나**:
  - 로그 변환: `np.log(<값>)`.
  - z-score: 평균을 빼고 표준편차로 나눕니다(`.mean()`·`.std(ddof=1)`).
  - Min-Max: 최솟값을 빼고 (최대−최소)로 나눕니다.
  - 왜도는 `stats.skew(<값>)`, 그림은 2-3 절의 `subplots` + `histplot`.
  - 군별 비교: 새로 만든 z 값을 컬럼으로 붙여(`df.assign(<이름>=…)`) `groupby(<군>)` 으로 평균.
- **이렇게 나오면 맞다**: **로그 변환만 왜도가 크게 줄고, z-score·Min-Max 는 왜도가 원본과 똑같이 남습니다.** 왜 그런지 한 줄로 설명할 수 있으면 이 미션의 핵심을 얻은 것입니다.
- **함정**: z-score·Min-Max 는 축의 **위치와 크기만 바꾸는 선형 변환**이라 분포 **모양**(치우침)은 그대로입니다 — "정규화하면 정규분포가 된다"는 오해를 여기서 깨야 합니다. `np.log` 는 0 이하에서 무한대·NaN 이 됩니다.

In [ ]:
# 여기에 코드를 작성하세요

### 선택 미션 4 (★★) — 이상치 고객·거래 탐색 전략
> 발제문 심화 분석 목표 ⑥

**비즈니스 질문**: "비정상적으로 큰" 거래와 고객을 어떻게 찾고, 찾은 다음 **제거할지 남길지** 어떻게 정하나?

- **무엇을 만드나**: IQR 1.5배 규칙으로 ① **거래 단위** 이상치와 ② **고객 총구매액 단위** 이상치를 각각 찾아, 건수·비율·**그들이 차지하는 매출 비중**을 출력합니다. 그리고 그 이상치를 **제거할지·남길지·따로 표시(플래그)할지**를 근거와 함께 한 줄로 적습니다.
- **어떤 도구를 쓰나**:
  - IQR 규칙: `.quantile([0.25, 0.75])` 로 Q1·Q3 → IQR → 상한은 Q3 + 1.5 × IQR.
  - 이상치만 고르기: 불리언 조건으로 필터 — 비율은 조건의 `.mean()`, 건수는 `.sum()`.
  - 고객 단위: `df.groupby('customer_id')['price'].sum()` 에 같은 규칙을 적용합니다.
  - 매출 비중: 이상치의 합 ÷ 전체 합.
  - 그림: `sns.boxplot(x=<시리즈>)` — 상자 밖 점이 이 규칙이 말하는 이상치입니다.
  - 어떤 상품군에서 나오는지 보려면 `value_counts().head()`.
- **이렇게 나오면 맞다**: 이상치 거래는 전체의 **5% 안팎인데 매출은 그보다 훨씬 큰 몫**을 차지합니다 — 그래서 "이상치니까 지운다"가 위험하다는 결론이 데이터에서 나옵니다. 거래 단위와 고객 단위의 결과가 다르다는 것도 확인하세요.
- **함정**: **이상치 = 오류가 아닙니다.** 여기서 잡힌 것은 대부분 정상적인 고가 상품(코트·신발)입니다 — 지우면 매출의 상당 부분이 사라집니다. 그리고 IQR 규칙은 **한쪽으로 치우친 분포에서 상단을 과하게 잡는다**는 한계가 있으니, 로그 변환 후 다시 보거나 상위 1% 같은 다른 기준과 비교해 보세요.

In [ ]:
# 여기에 코드를 작성하세요

### 선택 미션 5 (★★) — 월별·분기별 평균 매출 — 추세와 시즌성
> 발제문 심화 분석 목표 ⑤

**비즈니스 질문**: 매출이 특정 시기에 몰리는가? 그 급증을 "성장"이라고 불러도 되는가?

- **무엇을 만드나**: 월별·분기별로 **총매출·건수·평균가**를 집계해 표로 만들고, 월별 흐름을 꺾은선으로, 분기별 비교를 막대로 그립니다. 그리고 **급증한 달이 건수 때문인지 단가 때문인지** 구분해 한 줄로 적습니다.
- **어떤 도구를 쓰나**:
  - 분기 파생: 날짜형 컬럼의 `.dt.quarter` (1~4).
  - 집계: `groupby(<월 또는 분기>).agg(…)` 로 합계·건수·평균을 한 표에.
  - 그림: 2-3 절의 `lineplot`(추이) + `barplot`(분기 비교). 집계 결과는 `reset_index()` 후 넣습니다.
  - 최대·최소 달 찾기: `.idxmax()`·`.idxmin()`.
- **이렇게 나오면 맞다**: 총매출이 가장 큰 달과 평균가가 가장 큰 달이 **서로 다릅니다** — 그래서 "매출이 늘었다"를 **건수 증가**와 **단가 상승**으로 쪼개 설명할 수 있으면 성공입니다.
- **함정**: 이 데이터는 **1년뿐**이라 "증가 추세/감소 추세" 같은 장기 판단을 할 수 없습니다(작년 같은 달과 비교할 수 없으므로). 시즌 이벤트(블랙프라이데이·연말연시)로 인한 급증을 성장으로 단정하지 말라는 발제문 경고가 바로 이 지점입니다.

In [ ]:
# 여기에 코드를 작성하세요

### 선택 미션 6 (★★) — 채널 비중 해석 보정 — 비율(%)과 ARPU
> 발제문 심화 분석 목표 FAQ(표본 크기 차이 보정)

**비즈니스 질문**: 온라인이 오프라인보다 매출이 크다는 말은, 온라인 **기록이 더 많아서** 생긴 착시가 아닐까?

- **무엇을 만드나**: 채널별로 **건수·총매출·고객 수**를 구하고, 여기서 ① 건수 비중(%) ② 매출 비중(%) ③ **ARPU(고객당 평균 구매액)** ④ 고객당 구매 건수 ⑤ 거래당 평균가를 계산한 표를 만듭니다. 그리고 **표본 크기 차이를 보정한 뒤에도 결론이 유지되는지** 한 줄로 적습니다.
- **어떤 도구를 쓰나**:
  - 고객 수는 중복을 뺀 개수 — `agg(<이름>=('customer_id', 'nunique'))`.
  - 비중: 각 값을 `<컬럼>.sum()` 으로 나눕니다.
  - ARPU = 총매출 ÷ 고객 수, 고객당 건수 = 건수 ÷ 고객 수, 거래당 평균가 = 총매출 ÷ 건수.
  - 두 채널을 모두 쓴 고객 수: `df.groupby('customer_id')['sales_channel_id'].nunique()` 가 2 인 고객을 셉니다.
- **이렇게 나오면 맞다**: 채널별 ARPU 와 거래당 평균가를 나란히 놓았을 때, **"온라인 우세"가 단순히 기록 수 차이 때문이 아니라는 것(또는 그 반대)** 을 근거를 들어 말할 수 있으면 성공입니다.
- **함정**: **채널별 고객 수를 더하면 전체 고객 수보다 많습니다** — 두 채널을 모두 쓴 고객이 양쪽에 세어지기 때문입니다(그래서 ARPU 는 채널별로만 비교하고 합산하지 마세요). 그리고 이 데이터는 애초에 온라인 기록이 더 많이 수집된 표본이라, 보정 후에도 **"시장 전체에서 온라인이 우세하다"로 일반화할 수는 없습니다.**

In [ ]:
# 여기에 코드를 작성하세요

### 선택 미션 7 (★★★) — 고객별 총 구매액 상위 20% vs 하위 20%
> 발제문 심화 분석 목표 ②

**비즈니스 질문**: 우리 매출은 소수의 고객에게 얼마나 몰려 있고, 그 고객들은 무엇이 다른가?

- **무엇을 만드나**: 고객별 **총구매액·구매건수·평균구매액·온라인 비중**을 만든 뒤 총구매액으로 **5분위**로 나눠, **상위 20%와 하위 20%** 의 지표를 비교하는 표와 그림을 만듭니다. 두 그룹이 **전체 매출에서 차지하는 비중**도 함께 계산합니다.
- **어떤 도구를 쓰나**:
  - 고객 단위 표: `df.groupby('customer_id').agg(…)` — 온라인 비중은 "채널이 2인가"라는 **참/거짓 컬럼의 평균**으로 구할 수 있습니다(`df.assign(<이름>=<조건>)` 으로 먼저 컬럼을 만들면 편합니다).
  - 5분위 나누기: `pd.qcut(<시리즈>, q=5, labels=[…])` — 같은 **개수**로 나눕니다(`pd.cut` 은 같은 **폭**으로 나눔).
  - 그룹 비교: `groupby(<그룹 컬럼>, observed=True)[<지표들>].mean()`.
  - 매출 점유율: 그룹별 합계 ÷ 전체 합계.
  - 그림: 2-3 절의 `barplot` 으로 그룹별 지표 비교.
- **이렇게 나오면 맞다**: 상위 20% 가 전체 매출의 **40% 이상**을 차지하고, 하위 20% 는 10% 미만입니다. 그리고 상위 그룹은 평균구매액·구매건수·온라인 비중이 **모두** 더 높게 나옵니다 — 세 지표 중 어느 것이 가장 크게 벌어지는지 말할 수 있으면 성공입니다.
- **함정**: `qcut` 은 **같은 개수**로 나누므로, 총구매액이 같은 고객이 많으면 경계가 딱 20%가 아닐 수 있습니다. 그리고 이 5분위는 **총구매액으로 만든 것**이라 "상위 20%가 총구매액이 크다"는 것은 동어반복입니다 — 의미 있는 발견은 **평균구매액·건수·채널처럼 나누는 데 쓰지 않은 지표**에서 나옵니다.

In [ ]:
# 여기에 코드를 작성하세요

### 선택 미션 8 (★★★) — 월별 매출의 3개월 롤링 평균
> 발제문 심화 분석 목표 ③

**비즈니스 질문**: 월별 매출이 들쭉날쭉할 때, **노이즈를 걷어낸 흐름**은 어떤 모양인가?

- **무엇을 만드나**: 월별 총매출 표를 만들고 **3개월 롤링 평균** 컬럼을 추가해, 원래 선과 롤링 선을 **한 그림에 겹쳐** 그립니다. 앞의 두 달이 왜 비어 있는지, 롤링 창을 6개월로 늘리면 무엇이 달라지는지도 한 줄로 적습니다.
- **어떤 도구를 쓰나**:
  - 월별 집계: `df.groupby('month')['price'].sum()` — 그래프에 넣으려면 `reset_index(name=<이름>)`.
  - 롤링 평균: `<시리즈>.rolling(<창 크기>).mean()` — 창 크기 3 이면 3개월 이동 평균.
  - 겹쳐 그리기: `plt.plot(<x>, <y>, label=…)` 을 두 번 부르고 `plt.legend()`.
  - 월 눈금 고정: `plt.xticks(range(1, 13))`.
- **이렇게 나오면 맞다**: 롤링 선이 원래 선보다 **매끄럽고**, 1·2월 자리는 값이 없어 선이 3월부터 시작합니다. 두 선이 어긋나는 달을 짚어 "그 달만 튀었다"고 말할 수 있으면 성공입니다.
- **함정**: 롤링 평균은 **창 크기−1 개월만큼 뒤로 늦게 반응**합니다(그래서 앞부분이 NaN). 우리 데이터는 12개월뿐이라 6개월 창을 쓰면 점이 7개만 남아 추세라고 부르기 어렵습니다 — **3개월 창이 이 데이터에 맞는 선택**입니다. 그리고 월별 합계는 **그 달의 일수·표본 수**에 영향을 받으니, 필요하면 평균가와 함께 보세요.

In [ ]:
# 여기에 코드를 작성하세요

## 5. 관찰 정리 (서술)
**이 절에서 할 일**: 위에서 본 집계·그래프를 근거로 **한 문단짜리 인사이트**를 씁니다. "A 이다" 로 끝내지 말고, **다른 설명 가능성**(다른 변수가 진짜 원인일 수도 있다는 점)도 함께 짚으세요 — 관찰연구는 상관이지 인과가 아닙니다.

**쓰는 순서**: ① 어떤 집계·그래프를 봤나 → ② 거기서 무엇을 관찰했나 → ③ 다른 설명 가능성은 없는가 → ④ 우연인지 검정으로 확인하고 싶은 질문.

- 어떤 축(연령대·채널·상품군 등)에서 가장 뚜렷한 차이를 봤나요?
- 그 차이가 우연이 아니라고 자신 있게 말할 수 있나요, 아니면 검정으로 확인하고 싶은가요?
- 표본 크기가 작은 범주나, 온라인에 치우친 이 데이터의 한계를 함께 적었나요?

*(여기에 관찰과, 다른 설명 가능성과, 다음 파트로 넘길 질문을 서술하세요)*

> ✅ **여기까지 되면 통과**: 한 문단 서술에 관찰 + 다른 설명 가능성 + 다음 파트로 넘길 질문이 모두 있으면 됩니다.